# Logistic Regression with Hyperparameter Tuning for Emotion Classification
This notebook trains and evaluates a Logistic Regression model for emotion classification on the Dutch GoEmotions dataset using TF-IDF features. It also performs hyperparameter tuning via GridSearchCV to select the best regularization parameters.

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV


In [6]:
# Load training and test data
train_df = pd.read_csv("../data/dataset/processed/go_emotion_dutch.csv")
test_df = pd.read_csv("../data/group_4_url_1_transcript.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
test_df.head()


Train shape: (57100, 4)
Test shape: (1050, 7)


,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity
0,1900-01-01 00:00:00,1900-01-01 00:00:02,TV GELDERLAND 2021.,TV GELDERLAND 2021.,neutrality,neutral,neutral
1,1900-01-01 00:00:30,1900-01-01 00:00:32,Daar boven staat een kist.,There is a chest up there.,observation,neutral,neutral
2,1900-01-01 00:00:33,1900-01-01 00:00:35,Ik weet zeker dat ik ze alle vier gehad heb.,I am sure that I have had all four of them.,certainty,neutral,moderate
3,1900-01-01 00:00:37,1900-01-01 00:00:39,"O, kwaak!","Oh, quack!",exclamation,surprise,mild
4,1900-01-01 00:00:41,1900-01-01 00:00:44,Wat moet een mens zonder Ellie Lust?,What should a person do without Ellie Lust?,longing,sadness,mild


In [13]:
# Prepare text and labels
train_texts = train_df['Sentence'].astype(str).tolist()
test_texts = test_df['Sentence'].astype(str).tolist()

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(train_df['Emotion_core'].tolist() + test_df['Emotion_core'].tolist())

train_labels = label_encoder.transform(train_df['Emotion_core'].tolist())
test_labels = label_encoder.transform(test_df['Emotion_core'].tolist())

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)
print("Number of classes:", num_classes)


Classes: ['anger' 'disgust' 'fear' 'happiness' 'neutral' 'sadness' 'surprise']
Number of classes: 7


In [14]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

print("Train TF-IDF shape:", X_train.shape)
print("Test TF-IDF shape:", X_test.shape)


Train TF-IDF shape: (57100, 10000)
Test TF-IDF shape: (1050, 10000)


In [10]:
# Logistic Regression with GridSearchCV
clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                         solver='saga', multi_class='multinomial',
                         random_state=42)

# Parameter grid
params = {
    "C": [0.1, 1, 10],
    "penalty": ["l1", "l2"]
}

grid = GridSearchCV(clf, params, cv=5, scoring="f1_weighted", n_jobs=-1)
grid.fit(X_train, train_labels)

print("Best params:", grid.best_params_)
print("Best CV F1 (weighted):", grid.best_score_)

Best params: {'C': 1, 'penalty': 'l2'}
Best CV F1 (weighted): 0.42143654236711764


c:\Users\19102\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [15]:
# Logistic Regression with best parameters
clf = LogisticRegression(
    C=1,
    penalty="l2",
    solver="saga", 
    multi_class="multinomial",
    max_iter=10000,
    class_weight="balanced",
    random_state=42
)

# Train
clf.fit(X_train, train_labels)

# Predict
predictions = clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(test_labels, predictions)
f1 = f1_score(test_labels, predictions, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}\n")

print("Classification Report:")
print(classification_report(test_labels, predictions, target_names=label_encoder.classes_))

Test Accuracy: 0.4514
Test F1 Score: 0.4582

Classification Report:
              precision    recall  f1-score   support

       anger       0.12      0.05      0.08        37
     disgust       0.01      0.12      0.02        17
        fear       0.36      0.07      0.12        56
   happiness       0.70      0.20      0.31       198
     neutral       0.62      0.67      0.65       546
     sadness       0.36      0.39      0.38       104
    surprise       0.17      0.22      0.19        92

    accuracy                           0.45      1050
   macro avg       0.34      0.25      0.25      1050
weighted avg       0.53      0.45      0.46      1050



c:\Users\19102\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
